# Lesson 7.2 — Advanced Networking: Direct Connect, VPN & Hybrid Architectures

**Level: Advanced** | **Exam:** SAP-C02 Domain 1, Task 1.1

## Hybrid Connectivity Comparison

| Option | Setup Time | Bandwidth | Latency | Encrypted | Cost |
|--------|-----------|-----------|---------|-----------|------|
| **Site-to-Site VPN** | Minutes | ~1.25 Gbps | Variable (internet) | Yes (IPsec) | $0.05/hr |
| **Direct Connect 1G** | Weeks | 1 Gbps | Consistent | No (add yourself!) | ~$0.30/hr |
| **Direct Connect 10G** | Weeks | 10 Gbps | Consistent | No | ~$2.25/hr |

### Direct Connect Architecture Options
```
Option 1: DX → Virtual Private Gateway → ONE VPC
Option 2: DX → Direct Connect Gateway → MULTIPLE VPCs (different regions)
Option 3: DX → Transit Gateway → ALL VPCs in region + VPNs
```

### VPN as DR for Direct Connect
Always run VPN as backup to Direct Connect:
```
Primary path: On-prem → Direct Connect → AWS VPC (lower cost, stable)
Backup path:  On-prem → Site-to-Site VPN → AWS VPC (automatic failover)
```

### Direct Connect Resilience Levels
- **Non-redundant**: Single connection — 99.9% SLA (not recommended for production)
- **High Resilience**: 2 connections to 2 different DX locations — 99.99% SLA
- **Maximum Resilience**: 4 connections to 2 DX locations, 2 customer routers — highest SLA

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Inventory hybrid connectivity
import boto3

ec2 = boto3.client('ec2', region_name='us-east-1')

# VPN connections
vpn_connections = ec2.describe_vpn_connections()['VpnConnections']
print(f"Site-to-Site VPN Connections ({len(vpn_connections)}):")
for vpn in vpn_connections:
    print(f"  {vpn['VpnConnectionId']}  State: {vpn['State']}")
    for tunnel in vpn.get('VgwTelemetry', []):
        print(f"    Tunnel: {tunnel['OutsideIpAddress']}  {tunnel['Status']}  ({tunnel['StatusMessage']})")

if not vpn_connections:
    print("  No VPN connections found.")

# Customer Gateways (on-prem side of VPN)
cgws = ec2.describe_customer_gateways(Filters=[{'Name': 'state', 'Values': ['available']}])['CustomerGateways']
print(f"\nCustomer Gateways ({len(cgws)}):")
for cgw in cgws:
    print(f"  {cgw['CustomerGatewayId']}  BGP ASN: {cgw['BgpAsn']}  IP: {cgw['IpAddress']}")

try:
    dx = boto3.client('directconnect', region_name='us-east-1')
    dx_connections = dx.describe_connections()['connections']
    print(f"\nDirect Connect Connections ({len(dx_connections)}):")
    for conn in dx_connections:
        print(f"  {conn['connectionId']}  {conn['connectionName']}  {conn['connectionState']}  {conn['bandwidth']}")
except Exception as e:
    print(f"\nDirect Connect: {e}")

## Exercise

In [ ]:
# TODO: A company has these requirements:
# - 3 on-premises data centers
# - 10 AWS VPCs across us-east-1 and eu-west-1
# - Need 10 Gbps consistent bandwidth for database replication
# - Must have 99.99% connectivity SLA
# - Need encrypted connections (compliance requirement)
# - Hybrid DNS: EC2 must resolve on-prem hostnames
#
# Design the architecture:

architecture = {
    "primary_connection": "TODO",          # DX / VPN / DX + VPN backup
    "dx_configuration": "TODO",            # Single connection / High Resilience / Max Resilience
    "vpc_connectivity": "TODO",            # VGW / DX Gateway / Transit Gateway
    "encryption_method": "TODO",           # MACsec / VPN over DX / IPsec
    "dns_resolution": "TODO",              # Route 53 Resolver Inbound/Outbound Endpoints
    "justification": "TODO"
}

for k, v in architecture.items():
    print(f"{k:30}: {v}")

## Summary
- **VPN**: Fastest setup, internet-based, encrypted. Max ~1.25 Gbps. Use for dev, backup, small orgs.
- **Direct Connect**: Dedicated fiber, consistent performance. Weeks to set up. No built-in encryption.
- **Always encrypt DX traffic**: Use MACsec (Layer 2) or IPsec VPN over DX
- **Direct Connect Gateway**: One DX to multiple VPCs across regions
- **For production**: DX as primary + VPN as standby failover

---
**Next:** [Lesson 7.3 — Advanced Security Architecture: Cross-Account Access & SCP Design](lesson-7.3-advanced-security-arch.ipynb)